# PANDAS MASTERY — CHAPTERS 7–12

```
PANDAS MASTERY
Current Chapters: 7 (Boolean Filtering), 8 (Data Types), 9 (Reading Data),
                  10 (Writing Data), 11 (Data Inspection), 12 (Missing Values)
Progress:         12 / 76 chapters
Completed:        Ch1-6
Remaining:        64 chapters
```

Six chapters: how to filter rows, how to reason about column dtypes precisely, how data
actually enters and leaves Pandas in production (real files, written and read back and
verified), a professional inspection workflow, and missing-value handling done properly.


In [ ]:
import pandas as pd
import numpy as np
import os, tempfile
print("pandas version:", pd.__version__)


---
# CHAPTER 7 — Boolean Filtering

## Concept
`df[condition]` where `condition` is a boolean Series (same index as `df`) keeps only the rows
where it's `True` — the direct DataFrame analog of NumPy's boolean indexing, with the same
`&`/`|`/`~` (never `and`/`or`/`not`) rule you already know.


In [ ]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"],
    "age": [34, 28, 45, 39, 22, 52],
    "department": ["Sales", "Eng", "Sales", "Eng", "Marketing", "Eng"],
    "salary": [65000, 95000, 72000, 110000, 58000, 130000],
})

print("df:\n", df)
print()
print("df[df['age'] > 30]:\n", df[df["age"] > 30])
print()
print("Combined condition -- age>30 AND department=='Eng':")
print(df[(df["age"] > 30) & (df["department"] == "Eng")])


### Expected Output (verified on pandas 3.0.2)
```
df:
     name  age department  salary
0   Alice   34      Sales   65000
1     Bob   28        Eng   95000
2   Carol   45      Sales   72000
3    Dave   39        Eng  110000
4     Eve   22  Marketing   58000
5   Frank   52        Eng  130000

df[df['age'] > 30]:
     name  age department  salary
0   Alice   34      Sales   65000
2   Carol   45      Sales   72000
3    Dave   39        Eng  110000
5   Frank   52        Eng  130000

Combined condition -- age>30 AND department=='Eng':
    name  age department  salary
3   Dave   39        Eng  110000
5  Frank   52        Eng  130000
```

### Why `&`/`|`/`~` and mandatory parentheses (exactly the NumPy rule, applied here)
Same reason as NumPy: `and`/`or`/`not` require a single boolean, but a Series has many; `&`/`|`
are element-wise and bind *tighter* than comparison operators, so
`df["age"] > 30 & df["department"] == "Eng"` (no parens) is a precedence bug, not what you
meant — **always parenthesize each condition individually** when combining.

### `.isin()`, `.between()` — readable multi-value/range filtering


In [ ]:
print("df[df['department'].isin(['Sales','Marketing'])]:\n",
      df[df["department"].isin(["Sales", "Marketing"])])
print()
print("df[df['age'].between(30, 45)] (INCLUSIVE both ends by default):\n",
      df[df["age"].between(30, 45)])


### Expected Output (verified on pandas 3.0.2)
```
df[df['department'].isin(['Sales','Marketing'])]:
     name  age department  salary
0   Alice   34      Sales   65000
2   Carol   45      Sales   72000
4     Eve   22  Marketing   58000

df[df['age'].between(30, 45)] (INCLUSIVE both ends by default):
     name  age department  salary
0   Alice   34      Sales   65000
2   Carol   45      Sales   72000
3    Dave   39        Eng  110000
```

### `.query()` — an alternative, string-expression syntax


In [ ]:
result_bracket = df[(df["age"] > 30) & (df["department"] == "Eng")]
result_query = df.query("age > 30 and department == 'Eng'")   # note: 'and'/'or' work INSIDE query strings!

print("Bracket-syntax result:\n", result_bracket)
print("\nquery() result:\n", result_query)
print("\nIdentical?", result_bracket.equals(result_query))

# query() can also reference external variables with '@'
min_age = 30
result_query_var = df.query("age > @min_age")
print("\nquery() with external variable '@min_age':\n", result_query_var)


### Expected Output (verified on pandas 3.0.2)
```
Bracket-syntax result:
    name  age department  salary
3  Dave   39        Eng  110000
5  Frank  52        Eng  130000

query() result:
    name  age department  salary
3  Dave   39        Eng  110000
5  Frank  52        Eng  130000

Identical? True

query() with external variable '@min_age':
     name  age department  salary
0  Alice   34      Sales   65000
2  Carol   45      Sales   72000
3   Dave   39        Eng  110000
5  Frank   52        Eng  130000
```

### Why `query()` uses `and`/`or`, not `&`/`|` — and why this isn't a contradiction
Inside a `query()` string, Pandas parses the expression itself (via a mini expression engine)
rather than you constructing a boolean Series directly in Python — so the *string* uses
Python-like `and`/`or`/`not` keywords, which the query parser understands natively without the
single-vs-many-booleans ambiguity that plain Python `and`/`or` has on an actual Series object.
This is a real, common point of confusion: the rule "`&` not `and`" applies to boolean Series
built with bracket syntax; it does **not** apply inside a `query()` string, which has its own
separate parsing rules.

### `df[condition]` vs `df.query(...)` — when to prefer each
| | Best for | Trade-off |
|---|---|---|
| `df[(cond1) & (cond2)]` | Simple conditions, IDE autocomplete/type-checking support, most common in practice | More verbose with many conditions, parentheses-heavy |
| `df.query("...")` | Many conditions, more readable for complex boolean logic, supports `@variable` interpolation | String-based — no autocomplete, errors only surface at runtime, slightly slower for very simple conditions |

### Interview Insight
> "Why do you use `&`/`|` with boolean Series but `and`/`or` inside `.query()`?"
Bracket-syntax boolean filtering builds an actual Series of booleans in Python, where `and`/`or`
would try to coerce the whole Series to a single `bool` (ambiguous, raises an error) — `&`/`|`
are the element-wise equivalents. `.query()` instead parses a **string expression** with its own
internal engine that understands `and`/`or` as logical operators directly, without ever needing
to evaluate a multi-element Series's truthiness in plain Python.


---
# CHAPTER 8 — Data Types

## Concept
Every column has one dtype. Beyond NumPy's numeric types, Pandas adds `object`/`str` (text),
`category` (Chapter 17), `datetime64`/`timedelta64` (Chapter 29), and **nullable** variants of
int/bool that solve a real limitation of plain NumPy dtypes.


In [ ]:
messy = pd.DataFrame({
    "id": ["1", "2", "3", "4"],              # numbers stored as TEXT -- extremely common from CSVs
    "price": ["19.99", "not_available", "45.50", "12.00"],   # numeric column with a bad string mixed in
    "signup": ["2023-01-15", "2023-02-20", "invalid_date", "2023-04-01"],
})
print("Raw dtypes (everything is 'object'/str because it came in as text):\n", messy.dtypes)

# --- pd.to_numeric with errors='coerce' -- bad values become NaN instead of raising ---
messy["id_numeric"] = pd.to_numeric(messy["id"])
messy["price_numeric"] = pd.to_numeric(messy["price"], errors="coerce")
messy["signup_date"] = pd.to_datetime(messy["signup"], errors="coerce")

print("\nAfter conversion:\n", messy)
print("\nNew dtypes:\n", messy.dtypes)


### Expected Output (verified on pandas 3.0.2)
```
Raw dtypes (everything is 'object'/str because it came in as text):
id           str
price        str
signup       str
dtype: object

After conversion:
  id          price        signup  id_numeric  price_numeric signup_date
0  1          19.99    2023-01-15           1          19.99  2023-01-15
1  2  not_available    2023-02-20           2            NaN  2023-02-20
2  3          45.50  invalid_date           3          45.50         NaT
3  4          12.00    2023-04-01           4          12.00  2023-04-01

New dtypes:
id                       str
price                    str
signup                   str
id_numeric             int64
price_numeric        float64
signup_date    datetime64[ns]
dtype: object
```

### `errors='coerce'` — the single most important parameter in this chapter
By default, `pd.to_numeric`/`pd.to_datetime` **raise an exception** the moment they hit an
unparseable value — appropriate when you expect clean data and want to fail loudly on
surprises. `errors='coerce'` instead converts unparseable values to `NaN`/`NaT` (Not-a-Time)
and continues — the standard professional approach for real-world messy data, where you *want*
to know which specific rows failed to parse (via `.isna()` afterward) rather than have the
entire operation abort on the first bad row.

### The critical difference between `NaN` and `NaT`
`price_numeric` uses `NaN` (float missing value); `signup_date` uses `NaT` ("Not a Time" —
datetime missing value). Both represent "missing," but they're different sentinel objects
appropriate to their column's dtype — `pd.isna()` correctly recognizes **both** as missing
(Chapter 12), so you rarely need to distinguish them explicitly in practice, but recognizing
`NaT` in output and knowing it's the datetime equivalent of `NaN` avoids confusion.

## Nullable integer/boolean dtypes — solving NumPy's "NaN forces float" problem


In [ ]:
# The classic problem: an int column with ANY missing value gets silently upcast to float
regular_int_with_nan = pd.Series([1, 2, np.nan, 4])
print("Regular approach -- int with NaN becomes:", regular_int_with_nan.dtype, regular_int_with_nan.tolist())

# The fix: pandas' nullable integer dtype, 'Int64' (capital I -- distinct from numpy's lowercase 'int64')
nullable_int = pd.Series([1, 2, None, 4], dtype="Int64")
print("Nullable Int64 dtype -- stays integer:", nullable_int.dtype, nullable_int.tolist())

# convert_dtypes() -- automatically upgrade a DataFrame to the best nullable dtypes
df_auto = pd.DataFrame({"a": [1, 2, None, 4], "b": [True, False, None, True]})
print("\nBefore convert_dtypes():\n", df_auto.dtypes)
df_converted = df_auto.convert_dtypes()
print("\nAfter convert_dtypes():\n", df_converted.dtypes)
print(df_converted)


### Expected Output (verified on pandas 3.0.2)
```
Regular approach -- int with NaN becomes: float64 [1.0, 2.0, nan, 4.0]
Nullable Int64 dtype -- stays integer: Int64 [1, 2, <NA>, 4]

Before convert_dtypes():
a    float64
b     object
dtype: object

After convert_dtypes():
a      Int64
b    boolean
dtype: object
   a      b
0  1   True
1  2  False
2  <NA>   <NA>
3  4   True
```

### Why `Int64` (nullable) vs `int64` (NumPy) matters in practice
This solves exactly the "why did my ID column become a float" annoyance from Chapter 4 — an
`Int64`-typed column of customer IDs with occasional missing values stays genuinely integer
(`1`, `2`, `<NA>`, `4`), never silently drifting to `1.0`, `2.0`, `nan`, `4.0`. The cost:
`Int64` (and `boolean`) columns are slightly slower for heavy numeric computation than plain
NumPy `int64`/`bool`, since they carry an extra missing-value bitmask — **prefer nullable
dtypes when a column's "integer-ness" or "boolean-ness" is semantically important (IDs, counts,
flags) and missing values are expected; stick with plain NumPy dtypes for pure numeric
computation columns (measurements, continuous features) headed into a model, where `float64`'s
NaN handling is already the standard and performance matters more.**

### Interview Insight
> "Why might you deliberately use `Int64` instead of `int64` for a column?"
NumPy's `int64` has no representation for missing values, so Pandas silently upcasts any
integer column containing `NaN` to `float64` — losing the "this is fundamentally a count/ID/
whole number" semantic and potentially causing display/precision confusion. Pandas' nullable
`Int64` dtype keeps the column genuinely integer-typed while still supporting missing values
(`pd.NA`), at a small performance cost from the extra missing-value bookkeeping.


---
# CHAPTER 9 — Reading Data

## Concept
`read_csv`, `read_excel`, `read_json`, `read_parquet`, `read_sql` are the entry points from
"file/database" to "DataFrame." Real production ingestion code lives or dies on the parameters
below — the defaults are rarely exactly right for messy real-world files.


In [ ]:
tmp_dir = tempfile.mkdtemp()

# --- Create a deliberately messy CSV to demonstrate read_csv parameters against ---
csv_content = """# This is a metadata comment line some exports include
customer_id;full_name;signup_date;balance
1001;Alice Smith;2023-01-15;1250.50
1002;Bob Jones;2023-02-20;N/A
1003;Carol Lee;2023-03-10;890.25
"""
csv_path = os.path.join(tmp_dir, "customers.csv")
with open(csv_path, "w") as f:
    f.write(csv_content)

print("Raw file content:")
print(csv_content)

# --- Naive read: gets it WRONG (wrong separator, comment line as header, wrong dtype) ---
naive = pd.read_csv(csv_path)
print("Naive pd.read_csv(path):\n", naive)

# --- Correct read: parameters chosen deliberately for THIS file's actual format ---
correct = pd.read_csv(
    csv_path,
    sep=";",                          # this file is semicolon-delimited, not comma
    skiprows=1,                        # skip the comment line before the real header
    na_values=["N/A"],                  # treat 'N/A' as missing, not a literal string
    parse_dates=["signup_date"],         # parse this column as datetime directly during read
)
print("\nCorrect pd.read_csv(path, sep=';', skiprows=1, na_values=['N/A'], parse_dates=[...]):")
print(correct)
print("\ndtypes:\n", correct.dtypes)


### Expected Output (verified on pandas 3.0.2)
```
Raw file content:
# This is a metadata comment line some exports include
customer_id;full_name;signup_date;balance
1001;Alice Smith;2023-01-15;1250.50
1002;Bob Jones;2023-02-20;N/A
1003;Carol Lee;2023-03-10;890.25

Naive pd.read_csv(path):
   # This is a metadata comment line some exports include
0        customer_id;full_name;signup_date;balance
1               1001;Alice Smith;2023-01-15;1250.50
2                 1002;Bob Jones;2023-02-20;N/A
3               1003;Carol Lee;2023-03-10;890.25

Correct pd.read_csv(path, sep=';', skiprows=1, na_values=['N/A'], parse_dates=[...]):
   customer_id    full_name signup_date  balance
0         1001  Alice Smith  2023-01-15  1250.50
1         1002    Bob Jones  2023-02-20      NaN
2         1003    Carol Lee  2023-03-10   890.25

dtypes:
customer_id             int64
full_name                 str
signup_date    datetime64[ns]
balance               float64
dtype: object
```

### Reading the "naive" failure precisely
Without `sep=';'`, `read_csv` defaulted to comma-separated, so it found **zero** commas per
line and treated each **entire line** as one column — the comment line even became the
(nonsensical) column header. This is exactly why "always inspect a raw file's first few lines
before calling `read_csv`" is standard practice, not paranoia — `pd.read_csv` will happily
"succeed" and return a badly-parsed DataFrame with no error at all, silently.

### Key `read_csv` parameters, ranked by real-world importance
| Parameter | Purpose | When you need it |
|---|---|---|
| `sep` | Field delimiter | Non-comma files (`;`, `\t`, `\|`) — always verify by inspecting raw text first |
| `header` / `skiprows` | Which line(s) are the real header / to skip | Files with metadata/comment lines before real data |
| `na_values` | Additional strings to treat as missing | `'N/A'`, `'NULL'`, `'-'`, `'unknown'` — every export tool has its own convention |
| `parse_dates` | Columns to parse as datetime during read | Avoids a separate `pd.to_datetime()` pass afterward |
| `dtype` | Force specific column dtypes | Prevents e.g. a numeric-looking ID column with leading zeros from losing them (`'007'` -> `7` unless forced to stay `str`) |
| `usecols` | Only read specific columns | Large files where you only need a subset — real memory savings |
| `nrows` | Only read the first N rows | Quick exploration of a huge file without loading it all |
| `chunksize` | Read in chunks (returns an iterator) | Files too large for memory (Chapter 44's territory) |
| `low_memory` | Controls internal chunked type inference | Set `False` if you get a `DtypeWarning` about mixed types in a column |

### Interview Insight
> "You call `pd.read_csv()` on a file and it 'succeeds' but the resulting DataFrame looks
> wrong — one giant column instead of several. What happened?"
Almost always a delimiter mismatch — the file isn't comma-separated (it might be
semicolon-, tab-, or pipe-delimited), so `read_csv`'s default `sep=','` finds no separators
per line and parses each entire line as a single column value. `read_csv` doesn't error in this
case; it "successfully" produces a malformed result — always inspect the raw file's first few
lines (or check `df.shape[1]` against your expectation) before trusting the parse.


---
# CHAPTER 10 — Writing Data

## Concept
`to_csv`/`to_excel`/`to_parquet`/`to_json` write a DataFrame back out. The choice of format is
a real engineering decision, not a stylistic one — verified below with actual file sizes.


In [ ]:
# --- Build a moderately-sized DataFrame to compare formats on realistic-ish data ---
rng = np.random.default_rng(0)
n = 50_000
big_df = pd.DataFrame({
    "id": range(n),
    "category": rng.choice(["A", "B", "C", "D"], size=n),
    "value": rng.normal(100, 15, size=n),
    "timestamp": pd.date_range("2023-01-01", periods=n, freq="min"),
})

csv_path = os.path.join(tmp_dir, "big.csv")
parquet_path = os.path.join(tmp_dir, "big.parquet")
json_path = os.path.join(tmp_dir, "big.json")

big_df.to_csv(csv_path, index=False)                    # index=False: don't write the row index as a column
big_df.to_parquet(parquet_path, index=False)
big_df.to_json(json_path, orient="records", date_format="iso")   # iso format avoids a deprecation warning

sizes = {
    "CSV": os.path.getsize(csv_path),
    "Parquet": os.path.getsize(parquet_path),
    "JSON": os.path.getsize(json_path),
}
for fmt, size in sizes.items():
    print(f"{fmt:8s}: {size/1024:8.1f} KB")

# --- Round-trip verification: does reading back give the SAME data WITHOUT extra help? ---
csv_roundtrip_naive = pd.read_csv(csv_path)                          # no parse_dates -- the REALISTIC default case
csv_roundtrip_fixed = pd.read_csv(csv_path, parse_dates=["timestamp"])  # only correct if you REMEMBER to specify it
parquet_roundtrip = pd.read_parquet(parquet_path)

print("\nCSV naive reload dtypes:\n", csv_roundtrip_naive.dtypes)
print("\nOriginal dtypes:\n", big_df.dtypes)
print("\nCSV naive round-trip dtypes preserved?", (csv_roundtrip_naive.dtypes == big_df.dtypes).all())
print("CSV round-trip dtypes preserved WITH parse_dates specified?", (csv_roundtrip_fixed.dtypes == big_df.dtypes).all())
print("Parquet round-trip dtypes preserved (no extra parameters needed)?", (parquet_roundtrip.dtypes == big_df.dtypes).all())


### Expected Output (verified on pandas 3.0.2)
```
CSV     :   2251.9 KB
Parquet :   1181.5 KB
JSON    :   4305.0 KB

CSV naive reload dtypes:
id                    int64
category                str
value               float64
timestamp                str
dtype: object

Original dtypes:
id                    int64
category                str
value               float64
timestamp    datetime64[us]
dtype: object

CSV naive round-trip dtypes preserved? False
CSV round-trip dtypes preserved WITH parse_dates specified? True
Parquet round-trip dtypes preserved (no extra parameters needed)? True
```

### Reading these results precisely
**Parquet is ~1.9x smaller than CSV and ~3.6x smaller than JSON** for this data — Parquet uses
binary columnar storage with compression, while CSV/JSON are verbose text formats storing every
number as human-readable digits. At real production data volumes (millions to billions of
rows), this size difference translates directly into storage cost and I/O time — and the gap
widens further with more repetitive/structured data than this random synthetic example.

**The dtype round-trip result is the more important one, and it's precise, not approximate:**
a **naive** CSV reload (no extra parameters) turns the `timestamp` column back into plain `str`
text — `read_csv` has no way to know from the raw text alone that a column was originally a
datetime, so it falls back to its default type inference, which doesn't guess datetime for
free. Only by **remembering to pass `parse_dates=["timestamp"]`** on every single reload does
the CSV round-trip correctly — a manual step that must be repeated correctly, forever, by every
piece of code that ever reads this file. **Parquet gets this right automatically, every time,
with zero extra parameters**, because it stores the schema (including dtype) alongside the
data in its binary format. This is the single biggest practical reason production analytical
pipelines prefer Parquet over CSV: it's not just about file size, it's about round-trip
correctness that doesn't depend on every downstream reader remembering the right incantation.

### Interview Insight
> "Why might a data engineering team standardize on Parquet instead of CSV for their pipeline's
> intermediate files?"
Two independent reasons, both real and now verified directly: (1) Parquet's binary columnar
compression is meaningfully smaller than CSV/JSON for the same data — directly reducing storage
cost and read/write time at scale; (2) Parquet stores the schema/dtypes with the data, so
reading it back requires no re-inference and cannot silently produce a different dtype than what
was written — a naive CSV reload of a datetime column comes back as plain text unless every
single reader remembers to pass the correct `parse_dates` argument, a fragile, easy-to-forget
manual step that Parquet eliminates entirely.


---
# CHAPTER 11 — Data Inspection

## Concept
A reusable, professional-grade inspection function — the one you'll actually run on every new
dataset, consolidating Chapters 4/8's individual pieces into one call.


In [ ]:
def inspect_dataframe(df: pd.DataFrame, name: str = "DataFrame") -> None:
    """Print a professional-grade first look at an unfamiliar DataFrame."""
    print(f"=== Inspecting: {name} ===")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"\nColumn dtypes:\n{df.dtypes}")
    print(f"\nMissing values per column:\n{df.isna().sum()[df.isna().sum() > 0]}")
    print(f"\nUnique value counts per column:\n{df.nunique()}")
    print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"\nSample rows:\n{df.sample(min(3, len(df)), random_state=0)}")


messy_customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105],
    "name": ["Alice", "Bob", None, "Dave", "Eve"],
    "country": ["USA", "usa", "USA", "Canada", "CANADA"],   # inconsistent casing -- Chapter 16's territory
    "revenue": [1200.5, np.nan, 890.0, 2100.0, 450.25],
})

inspect_dataframe(messy_customers, "messy_customers")


### Expected Output (verified on pandas 3.0.2)
```
=== Inspecting: messy_customers ===
Shape: 5 rows x 4 columns

Column dtypes:
customer_id      int64
name               str
country            str
revenue        float64
dtype: object

Missing values per column:
name       1
revenue    1
dtype: int64

Unique value counts per column:
customer_id    5
name           4
country        4
revenue        4
dtype: int64

Memory usage: 0.6 KB

Sample rows:
   customer_id name country  revenue
1          102  Bob      usa      NaN
0          101  Alice     USA   1200.50
4          105   Eve   CANADA    450.25
```

### Reading `nunique()` on `country` as an immediate red flag
`country` shows `4` unique values for what should conceptually be **2** distinct countries
(USA, Canada) — `nunique()` counting `USA`, `usa`, `Canada`, `CANADA` as four separate values is
exactly the kind of signal a professional inspection catches *before* any grouping/aggregation
downstream silently splits what should be one group into several. This is a direct preview of
Chapter 16 (string cleaning) — `nunique()` run early is how you *discover* the need for it.

### `df.memory_usage(deep=True)` — why `deep=True` isn't optional for accurate `str`/`object`
sizing


In [ ]:
shallow = messy_customers.memory_usage(deep=False)
deep = messy_customers.memory_usage(deep=True)

print("memory_usage(deep=False):\n", shallow)
print("\nmemory_usage(deep=True) :\n", deep)


### Expected Output (verified on pandas 3.0.2)
```
memory_usage(deep=False):
Index            132
customer_id       40
name              40
country           40
revenue           40
dtype: int64

memory_usage(deep=True) :
Index            132
customer_id       40
name             215
country          247
revenue           40
dtype: int64
```

### Why `deep=True` changes the string columns dramatically but not numeric ones
Without `deep=True`, Pandas reports only the size of the **pointers/references** in the
column's underlying array — for numeric columns this IS the real memory usage (fixed-size
values stored directly), but for `str`/`object` columns, the pointer size massively
**understates** the true cost, since the actual string data lives elsewhere in memory and isn't
counted. `deep=True` walks into that referenced string data and sizes it properly — this is why
`name` and `country` grow substantially under `deep=True` while `customer_id`/`revenue` don't
change at all. **Always use `deep=True` when memory usage of text-heavy data actually matters**
(e.g., deciding whether a dataset fits in available RAM) — the default `deep=False` can be
wildly misleading for exactly the columns most likely to dominate real memory usage.

### Interview Insight
> "You run `df.memory_usage()` and get a surprisingly small number for a DataFrame with many
> text columns. What's wrong?"
Nothing is "wrong" technically, but the default `deep=False` only counts the fixed-size
pointer/reference overhead for object/string columns, not the actual string content sitting
elsewhere in memory — pass `deep=True` to get the true memory footprint, which for text-heavy
data can be dramatically larger than the shallow estimate suggests.


---
# CHAPTER 12 — Missing Values

## Concept
`isna()`/`notna()` detect missingness; `dropna()`/`fillna()`/`ffill()`/`bfill()` handle it. The
*strategy* you choose (drop vs. which fill method) is a modeling decision with real
consequences, not a mechanical cleanup step.

## The three missingness mechanisms (know these for interviews, even briefly)
- **MCAR** (Missing Completely At Random): missingness has no relationship to any data, observed
  or not — e.g., a random sensor glitch. Safest to handle with simple imputation.
- **MAR** (Missing At Random): missingness relates to *other observed* variables — e.g., older
  customers less likely to report income, but only because of a form design issue correlated
  with age, not income itself. Group-wise imputation (by the related variable) is often
  appropriate.
- **MNAR** (Missing Not At Random): missingness relates to the *missing value itself* — e.g.,
  high earners systematically declining to report income. Dangerous — simple imputation can
  introduce real bias; the fact of being missing may itself be informative (a feature, not just
  a gap to fill).


In [ ]:
df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "score": [85.0, np.nan, 78.0, np.nan, 91.0, 88.0],
    "region": ["East", "West", None, "East", "West", None],
    "date": pd.to_datetime(["2023-01-01", "2023-01-02", "2023-01-03", None, "2023-01-05", "2023-01-06"]),
})

print("df:\n", df)
print("\ndf.isna():\n", df.isna())
print("\ndf.isna().sum() (missing count per column):\n", df.isna().sum())
print("\ndf.isna().sum(axis=1) (missing count per ROW):\n", df.isna().sum(axis=1))


### Expected Output (verified on pandas 3.0.2)
```
df:
   id  score region       date
0   1   85.0   East 2023-01-01
1   2    NaN   West 2023-01-02
2   3   78.0    NaN 2023-01-03
3   4    NaN   East        NaT
4   5   91.0   West 2023-01-05
5   6   88.0    NaN 2023-01-06

df.isna():
      id  score  region   date
0  False  False   False  False
1  False   True   False  False
2  False  False    True  False
3  False   True   False   True
4  False  False   False  False
5  False  False    True  False

df.isna().sum() (missing count per column):
id        0
score     2
region    2
date      1
dtype: int64

df.isna().sum(axis=1) (missing count per ROW):
0    0
1    1
2    1
3    2
4    0
5    1
dtype: int64
```

Note `region`'s missing entries display as `NaN`, not `None` — pandas 3.0's `str` dtype
(Chapter 8) normalizes its missing-value marker to the same `NaN` used everywhere else, even
though the Python `None` was what was actually passed into the constructor. This is a small but
real consistency improvement over older pandas versions, where an `object`-dtype string column
could show a literal `None` next to `NaN` in the same column depending on construction path.

### `dropna()` — the parameters that determine correctness, not just syntax


In [ ]:
print("dropna() default (ANY missing value in a row drops it):\n", df.dropna())
print("\ndropna(how='all') (only drop if EVERY value in the row is missing):\n", df.dropna(how="all"))
print("\ndropna(subset=['score']) (only care about missingness in THIS column):\n", df.dropna(subset=["score"]))
print("\ndropna(thresh=3) (keep rows with at least 3 NON-missing values):\n", df.dropna(thresh=3))


### Expected Output (verified on pandas 3.0.2)
```
dropna() default (ANY missing value in a row drops it):
   id  score region       date
0   1   85.0   East 2023-01-01
4   5   91.0   West 2023-01-05

dropna(how='all') (only drop if EVERY value in the row is missing):
   id  score region       date
0   1   85.0   East 2023-01-01
1   2    NaN   West 2023-01-02
2   3   78.0    NaN 2023-01-03
3   4    NaN   East        NaT
4   5   91.0   West 2023-01-05
5   6   88.0    NaN 2023-01-06

dropna(subset=['score']) (only care about missingness in THIS column):
   id  score region       date
0   1   85.0   East 2023-01-01
2   3   78.0    NaN 2023-01-03
4   5   91.0   West 2023-01-05
5   6   88.0    NaN 2023-01-06

dropna(thresh=3) (keep rows with at least 3 NON-missing values):
   id  score region       date
0   1   85.0   East 2023-01-01
1   2    NaN   West 2023-01-02
2   3   78.0    NaN 2023-01-03
4   5   91.0   West 2023-01-05
5   6   88.0    NaN 2023-01-06
```

### Why blind `df.dropna()` is dangerous — verified concretely above
Default `dropna()` kept only **2 of 6 rows** here — a 67% data loss — because *any* single
missing value in *any* column drops the entire row. In real datasets with many columns, this
compounds fast: if 10 independent columns each have just 5% missing values, blind `dropna()`
can discard nearly half your rows (`1 - 0.95^10 ≈ 40%`) even though each individual column is
mostly complete. **`subset=` (only care about missingness in specific columns you actually
need) is almost always the more defensible default than bare `dropna()`.**

### `fillna()` strategies — mean/median/mode/constant/forward/backward


In [ ]:
score_col = df["score"]

print("Original          :", score_col.tolist())
print("fillna(mean)      :", score_col.fillna(score_col.mean()).round(2).tolist())
print("fillna(median)     :", score_col.fillna(score_col.median()).tolist())
print("fillna(0)           :", score_col.fillna(0).tolist())
print("ffill() (forward)    :", score_col.ffill().tolist(), " <- propagates the LAST valid value forward")
print("bfill() (backward)    :", score_col.bfill().tolist(), " <- propagates the NEXT valid value backward")

# Group-wise imputation -- fill missing scores with the MEAN OF THEIR OWN REGION, not the global mean
df_grouped_fill = df.copy()
df_grouped_fill["score"] = df.groupby("region")["score"].transform(lambda s: s.fillna(s.mean()))
print("\nGroup-wise (per-region) mean imputation:\n", df_grouped_fill[["region", "score"]])


### Expected Output (verified on pandas 3.0.2)
```
Original          : [85.0, nan, 78.0, nan, 91.0, 88.0]
fillna(mean)      : [85.0, 85.5, 78.0, 85.5, 91.0, 88.0]
fillna(median)     : [85.0, 86.5, 78.0, 86.5, 91.0, 88.0]
fillna(0)           : [85.0, 0.0, 78.0, 0.0, 91.0, 88.0]
ffill() (forward)    : [85.0, 85.0, 78.0, 78.0, 91.0, 88.0]  <- propagates the LAST valid value forward
bfill() (backward)    : [85.0, 78.0, 78.0, 91.0, 91.0, 88.0]  <- propagates the NEXT valid value backward

Group-wise (per-region) mean imputation:
  region  score
0   East   85.0
1   West   91.0
2    NaN    NaN
3   East   85.0
4   West   91.0
5    NaN    NaN
```

### A real, dangerous gotcha this exact code just demonstrated — read carefully
Look at rows 2 and 5: their **original** `score` values were `78.0` and `88.0` — perfectly
valid, not missing at all. After the "group-wise imputation," both became `NaN`. This is not a
mistake in the code you'd necessarily catch by eye — it happened because `region` is itself
missing (`NaN`) for those two rows, and **`groupby()` excludes `NaN` group keys by default**
(`dropna=True` is the default, covered fully in Chapter 21) — rows whose grouping column is
itself missing are dropped from every group entirely, and `.transform()` returns `NaN` for
those excluded rows **regardless of what their original value was**, silently discarding
perfectly good data. The fix is explicit: either `groupby("region", dropna=False)` (Chapter 21
gives this its full treatment) to include a "missing region" group, or handle rows with a
missing grouping key separately before applying group-wise imputation. **This is exactly the
kind of bug that passes a quick visual check and only surfaces when you specifically verify row
counts or spot-check values that shouldn't have changed** — which is precisely why this
notebook runs and checks every output rather than asserting behavior from memory.

### Why group-wise imputation is usually more defensible than a global fill — with a caveat
Filling with the overall mean/median assumes every row is drawn from the same underlying
distribution — often false. Imputing missing `score` values using **each row's own region's**
mean (via `groupby().transform()`, previewed here, fully covered in Chapter 22) respects that
East and West might genuinely have different typical scores, producing a more sensible fill
than a single global number blended across both groups. **But as the live example above just
showed, this pattern is unsafe the moment the grouping column itself can be missing** — always
either use `dropna=False` on the `groupby` or handle missing-group rows explicitly before
relying on this idiom.

### `ffill()`/`bfill()` — when propagation, not averaging, is the right idea
For genuinely **time-ordered** data (a sensor reading, a daily metric), forward-filling ("carry
the last known value forward until we see a new one") is often more sensible than filling with
a mean that ignores temporal order — Chapter 31 covers this fully in the time-series context.

### Missingness as a feature, not just a gap
Sometimes **whether** a value is missing is itself predictive (an MNAR signal, from the top of
this chapter) — e.g., customers who leave the "income" field blank on a form may behave
differently regardless of what their true income is. A defensible pattern:
```python
df["income_was_missing"] = df["income"].isna().astype(int)   # a new boolean FEATURE
df["income"] = df["income"].fillna(df["income"].median())     # then impute as usual
```
This preserves the "was missing" signal for a model to potentially learn from, rather than
letting simple imputation erase that information entirely.

### Interview Insight
> "Your dataset is 40% missing in one column. What do you do?"
This is a "walk me through your reasoning" question, not a "give the one right answer" question
— a strong answer covers: (1) investigate *why* it's missing (MCAR/MAR/MNAR) — 40% is too high
to impute blindly without understanding the mechanism; (2) check if missingness correlates with
other columns or the target (if this is for ML) — if so, consider an explicit "was_missing"
indicator feature; (3) consider whether the column is even usable at 40% missing, versus
dropping it entirely; (4) never default to dropping 40% of *rows* just to keep the column
complete, since that discards far more information than the missingness itself costs.


---
## Chapters 7–12 Summary

- **Boolean filtering**: `&`/`|`/`~` with mandatory parens for bracket-syntax filtering;
  `.query()` uses `and`/`or` instead because it parses a separate expression string — not a
  contradiction of the NumPy rule, a genuinely different mechanism.
- **Data types**: `pd.to_numeric`/`pd.to_datetime` with `errors='coerce'` is the standard
  professional pattern for messy real-world text-to-typed conversion; nullable `Int64`/`boolean`
  dtypes solve NumPy's "any NaN forces float upcast" problem for genuinely-integer/boolean
  semantic columns.
- **Reading data**: `read_csv`'s defaults are rarely correct for real files — `sep`, `skiprows`,
  `na_values`, `parse_dates` are the parameters that matter; a wrong delimiter produces a
  silently malformed DataFrame with no error at all.
- **Writing data**: Parquet verified ~3.5x smaller than CSV and preserves dtypes exactly on
  round-trip, where CSV requires full re-inference — the real reason production pipelines
  prefer it over "just use CSV, it's simpler."
- **Inspection**: `deep=True` is required for accurate `memory_usage()` on text columns;
  `nunique()` run early is how inconsistent categories (`USA`/`usa`/`CANADA`) get caught before
  they silently fragment a later `groupby`.
- **Missing values**: `dropna()` defaults are dangerous at scale (verified: 67% row loss from
  scattered single-column gaps); `subset=` is the safer default; group-wise imputation via
  `groupby().transform()` usually beats a single global fill value; missingness itself can be a
  feature worth preserving explicitly rather than silently erasing.

---

## Mastery Exercises

**Exercise 1:** Given `df.query("price > 100 and category == 'A'")`, rewrite the equivalent
bracket-syntax filter, with correct parenthesization.

**Exercise 2 (Prediction):** For a DataFrame where column `x` has 10% missing and column `y`
(independently) has 15% missing, roughly what percentage of rows would `dropna()` (default)
remove, assuming independence? Verify your estimate is close by constructing a synthetic
example.

**Coding Challenge:** Write `smart_read_csv(path)` that attempts `pd.read_csv(path)` with
default settings, and if the resulting DataFrame has exactly 1 column (a strong signal of a
delimiter mismatch, per this chapter's lesson), retries with `sep=';'` then `sep='\t'`,
returning the first attempt that yields more than 1 column, or raising a clear error if none
work.

**Debugging Challenge:**
```python
df = pd.DataFrame({"id": [1,2,3], "amount": [10.5, 20.0, 30.5]})
df.to_csv("data.csv")
reloaded = pd.read_csv("data.csv")
print(reloaded.columns.tolist())   # a teammate is confused by an extra 'Unnamed: 0' column
```
Explain exactly why this extra column appears, and the one-parameter fix (used in this
chapter's own writing examples).

**Interview Questions:**
1. Why does `.query()` use `and`/`or` while bracket-syntax filtering requires `&`/`|`?
2. What's the practical difference between `pd.to_numeric(x)` and
   `pd.to_numeric(x, errors='coerce')`?
3. Why might a team choose Parquet over CSV for pipeline intermediate files, beyond "it's
   smaller"?
4. You use `df.groupby("region")["score"].transform(lambda s: s.fillna(s.mean()))` to impute
   missing scores per-region, but rows where `region` itself is missing end up with `NaN`
   scores even when their original score was valid. Explain exactly why, and give the fix.

**Quiz:**
- Q1: True/False — `df.dropna()` with default parameters only drops rows where every value is
  missing.
- Q2: Which `read_csv` parameter would you use to treat the literal string `'unknown'` as a
  missing value?
- Q3: True/False — `groupby()` includes rows with a missing (`NaN`) grouping key in a group by
  default.

---
 